### **Chunk 10: The Residual Connection (ResNet) - The Crucible**

**The Challenge:** Our ResNet-18, engineered in the clean environment of CIFAR-10, performed well. But was that because the model is genuinely powerful, or because the task was relatively simple? The Crucible will answer this.

*   **Concept:** Stress-testing the ResNet-18's ability to learn fine-grained features.
*   **Dataset:** `CIFAR-100`. This dataset has the **same number of images** (50,000 train, 10,000 test) and the **same image size** (32x32) as CIFAR-10, but with **100 classes** instead of 10.
*   **The Crucible Effect:** With only 500 training images per class (down from 5000), the model is starved for data. It can no longer succeed by learning simple features. It must learn to distinguish between highly similar classes (e.g., "oak tree" vs. "maple tree," "otter" vs. "beaver"). This dramatically increases the risk of overfitting and exposes the true generalization power of our architecture.
*   **Goal:** Adapt our existing code to handle 100 classes, train the model, and then perform a deep, glass-box analysis to understand its new failure modes in a data-scarce, fine-grained environment.

In [2]:
# --- Imports ---
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# --- Matplotlib and Seaborn Settings ---
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("talk")

-----

### Load CIFAR-100 Dataset 

In [3]:
# We will download it into its own sub-directory within the Crucible's data folder.
train_dataset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True)
test_dataset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True)

# --- Load Class Names ---
# The labels in CIFAR-100 are just indices 0-99. We need to map them to actual names.
# Source: https://www.cs.toronto.edu/~kriz/cifar.html
cifar100_classes = [
    'apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 
    'bicycle', 'bottle', 'bowl', 'boy', 'bridge', 'bus', 'butterfly', 'camel', 
    'can', 'castle', 'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock', 
    'cloud', 'cockroach', 'couch', 'crab', 'crocodile', 'cup', 'dinosaur', 
    'dolphin', 'elephant', 'flatfish', 'forest', 'fox', 'girl', 'hamster', 
    'house', 'kangaroo', 'keyboard', 'lamp', 'lawn_mower', 'leopard', 'lion',
    'lizard', 'lobster', 'man', 'maple_tree', 'motorcycle', 'mountain', 'mouse',
    'mushroom', 'oak_tree', 'orange', 'orchid', 'otter', 'palm_tree', 'pear',
    'pickup_truck', 'pine_tree', 'plain', 'plate', 'poppy', 'porcupine',
    'possum', 'rabbit', 'raccoon', 'ray', 'road', 'rocket', 'rose',
    'sea', 'seal', 'shark', 'shrew', 'skunk', 'skyscraper', 'snail', 'snake',
    'spider', 'squirrel', 'streetcar', 'sunflower', 'sweet_pepper', 'table',
    'tank', 'telephone', 'television', 'tiger', 'tractor', 'train', 'trout',
    'tulip', 'turtle', 'wardrobe', 'whale', 'willow_tree', 'wolf', 'woman',
    'worm'
]